<a href="https://colab.research.google.com/github/sreenadhyadav883/7006SCN2526MAYJUL-7006SCN_SYA_16073319/blob/main/notebooks/Task5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
# Run this cell once at the start of a fresh Colab runtime
!apt-get update -qq
!apt-get install -y openjdk-17-jdk-headless -qq > /dev/null
!pip install -q -U "pyspark[connect]~=4.0.0" findspark

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [10]:
import os
import findspark

# Java path used by Colab after installing OpenJDK 17
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
findspark.init()
from pyspark.sql import SparkSession

# Stop an old Spark session if the cell is re-run
try:
    spark.stop()
except NameError:
    pass
spark = (
    SparkSession.builder
    .appName("7006SCN_PySpark")
    .master("local[2]")
    .config("spark.driver.memory", "2g")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)

spark.conf.set("spark.sql.repl.eagerEval.enabled", True)
spark

In [3]:
# Loading the dataset from the public mirror
# Used -q (quiet) so it doesn't spam the googlecolab output with download logs
!wget -q https://datasets-documentation.s3.eu-west-3.amazonaws.com/amazon_reviews/amazon_reviews_2015.snappy.parquet

In [11]:
import time
from pyspark.sql.functions import col, when
from pyspark.sql.types import StringType
from pyspark.ml.feature import Tokenizer, StopWordsRemover, HashingTF, IDF
from pyspark.ml import Pipeline


df = spark.read.parquet("amazon_reviews_2015.snappy.parquet")
clean_df = df.dropna(subset=["review_body", "star_rating"])
clean_df = clean_df.withColumn("review_body", col("review_body").cast("string"))
final_df = clean_df.withColumn("label", when(col("star_rating") >= 4, 1).otherwise(0))

#Down-sample to 1% (approx 420,000 rows) so that google colab 12gb memory dont crash and throw a "Out of Memory" Error
sampled_df = final_df.sample(withReplacement=False, fraction=0.01, seed=42)
print(f"Sampled Dataset Size for Model Training: {sampled_df.count():,} rows")

#Reducing max features from 10,000 to 5,000
tokenizer = Tokenizer(inputCol="review_body", outputCol="words")
remover = StopWordsRemover(inputCol="words", outputCol="filtered_words")
hashing_tf = HashingTF(inputCol="filtered_words", outputCol="rawFeatures", numFeatures=5000)
idf = IDF(inputCol="rawFeatures", outputCol="features")
pipeline = Pipeline(stages=[tokenizer, remover, hashing_tf, idf])
pipeline_model = pipeline.fit(sampled_df)
ml_data = pipeline_model.transform(sampled_df)


#Train/Test Split of 80:20 ratio
train_data, test_data = ml_data.randomSplit([0.8, 0.2], seed=42)
#Cache the training data in memory to dramatically speed up Cross-Validation
train_data.cache()
test_data.cache()



Sampled Dataset Size for Model Training: 419,167 rows


review_date,marketplace,customer_id,review_id,product_id,product_parent,product_title,product_category,star_rating,helpful_votes,total_votes,vine,verified_purchase,review_headline,review_body,label,words,filtered_words,rawFeatures,features
16455,[55 53],728931,[52 33 54 50 4E 5...,[42 30 30 35 31 4...,502709431,[53 65 74 74 6F 6...,[47 72 6F 63 65 7...,5,5,7,false,true,[46 69 76 65 20 5...,Very good flavor.,1,"[very, good, flav...","[good, flavor.]","(5000,[1168,2752]...","(5000,[1168,2752]..."
16455,[55 53],953223,[52 31 57 4A 37 4...,[42 30 30 48 5A 4...,793336716,[4D 61 64 64 20 4...,[4F 75 74 64 6F 6...,4,0,1,false,false,[69 74 73 20 61 6...,My headset creaks...,1,"[my, headset, cre...","[headset, creaks,...","(5000,[44,115,466...","(5000,[44,115,466..."
16455,[55 53],1108406,[52 32 46 42 47 5...,[42 30 30 4B 4C 4...,933496237,[53 70 6F 74 69 6...,[4D 6F 62 69 6C 6...,5,0,0,false,true,[41 77 65 73 6F 6...,Let's you hear fu...,1,"[let's, you, hear...","[hear, full, albu...","(5000,[521,794,93...","(5000,[521,794,93..."
16455,[55 53],1190686,[52 31 50 58 54 5...,[42 30 30 4E 47 5...,407911167,[55 62 65 72 76 6...,[48 65 61 6C 74 6...,5,0,0,false,false,[4D 6F 72 65 20 6...,Combined with the...,1,"[combined, with, ...","[combined, testos...","(5000,[276,817,11...","(5000,[276,817,11..."
16455,[55 53],1294414,[52 31 52 50 50 5...,[42 30 30 46 4A 5...,714608589,[42 61 6E 64 69 7...,[4F 75 74 64 6F 6...,5,0,2,false,true,[46 69 76 65 20 5...,Works great .,1,"[works, great, .]","[works, great, .]","(5000,[750,1828,2...","(5000,[750,1828,2..."
16455,[55 53],1355719,[52 31 57 47 4F 4...,[42 30 30 48 4C 5...,19854895,[4D 61 67 69 63 2...,[4B 69 74 63 68 6...,5,0,0,false,true,[46 69 76 65 20 5...,Very good value. ...,1,"[very, good, valu...","[good, value., wi...","(5000,[490,495,78...","(5000,[490,495,78..."
16455,[55 53],1776042,[52 31 4F 52 4A 4...,[42 30 30 33 31 4...,638051363,[4E 49 4E 45 53 5...,[48 6F 6D 65],5,0,0,false,false,[48 61 6E 64 79],Started using it ...,1,"[started, using, ...","[started, using, ...","(5000,[114,170,44...","(5000,[114,170,44..."
16455,[55 53],2112267,[52 33 54 45 48 4...,[42 30 30 46 51 4...,940671580,[56 69 76 61 20 4...,[48 65 61 6C 74 6...,5,0,0,false,false,[46 69 76 65 20 5...,Very Good Excelle...,1,"[very, good, exce...","[good, excellent<...","(5000,[33,245,116...","(5000,[33,245,116..."
16455,[55 53],3036122,[52 32 33 42 46 4...,[42 30 30 43 54 5...,617043120,[54 68 65 20 53 6...,[4D 6F 62 69 6C 6...,1,0,1,false,true,[4F 6E 65 20 53 7...,Claimed you could...,0,"[claimed, you, co...","[claimed, design,...","(5000,[320,942,10...","(5000,[320,942,10..."
16455,[55 53],3036122,[52 33 39 31 34 5...,[42 30 30 46 45 4...,909902855,[47 72 65 65 6E 6...,[4D 6F 62 69 6C 6...,1,0,0,false,true,[4F 6E 65 20 53 7...,Waste of time,0,"[waste, of, time]","[waste, time]","(5000,[3157,4012]...","(5000,[3157,4012]..."


In [13]:
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.evaluation import BinaryClassificationEvaluator,MulticlassClassificationEvaluator
from pyspark.ml.classification import NaiveBayes
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.classification import LinearSVC
from pyspark.mllib.evaluation import MulticlassMetrics

lr_model = LogisticRegression(featuresCol="features", labelCol="label", regParam=0.1).fit(train_data)
nb_model = NaiveBayes(featuresCol="features", labelCol="label", smoothing=0.5).fit(train_data)
rf_model = RandomForestClassifier(featuresCol="features", labelCol="label", numTrees=20, seed=42).fit(train_data)
lsvc_model = LinearSVC(featuresCol="features", labelCol="label", regParam=0.01).fit(train_data)

models = {
    "Logistic Regression": lr_model,
    "Naive Bayes": nb_model,
    "Random Forest": rf_model,
    "Linear SVC": lsvc_model
}

eval_roc = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC")
eval_pr = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderPR")
eval_multi = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction")

print("\nMODEL EVALUATION RESULTS")

for name, model in models.items():
    preds = model.transform(test_data)

    roc_val = eval_roc.evaluate(preds)
    pr_val = eval_pr.evaluate(preds)
    f1_val = eval_multi.evaluate(preds, {eval_multi.metricName: "f1"})
    acc_val = eval_multi.evaluate(preds, {eval_multi.metricName: "accuracy"})

    print("\nModel: " + name)
    print("ROC AUC: " + str(round(roc_val, 4)))
    print("PR AUC: " + str(round(pr_val, 4)))
    print("F1 Score: " + str(round(f1_val, 4)))
    print("Accuracy: " + str(round(acc_val, 4)))

    # Confusion Matrix
    preds_rdd = preds.select("prediction", "label").rdd.map(lambda row: (float(row.prediction), float(row.label)))
    matrix = MulticlassMetrics(preds_rdd).confusionMatrix().toArray()
    print("Confusion Matrix:")
    print(matrix)



MODEL EVALUATION RESULTS

Model: Logistic Regression
ROC AUC: 0.8673
PR AUC: 0.9548
F1 Score: 0.789
Accuracy: 0.8316
Confusion Matrix:
[[ 3371. 13200.]
 [  900. 66269.]]

Model: Naive Bayes
ROC AUC: 0.5917
PR AUC: 0.8539
F1 Score: 0.8307
Accuracy: 0.8239
Confusion Matrix:
[[11199.  5372.]
 [ 9373. 57796.]]

Model: Random Forest
ROC AUC: 0.7424
PR AUC: 0.9194
F1 Score: 0.714
Accuracy: 0.8021
Confusion Matrix:
[[    0. 16571.]
 [    0. 67169.]]

Model: Linear SVC
ROC AUC: 0.8585
PR AUC: 0.9485
F1 Score: 0.8288
Accuracy: 0.8506
Confusion Matrix:
[[ 6000. 10571.]
 [ 1939. 65230.]]
